# ELSA Depression Prediction Pipeline
## Predicting Wave 8 Depression from Wave 6 & 7 Predictors

**Authors:** Fiyin Akano, Zannat Chowdhury Sagar  
**Module:** MSc AI & Healthcare — University of Surrey  
**Date:** April 2026

---

### Design

| Element | Choice | Rationale |
|---------|--------|-----------|
| **Task** | Binary classification | "Will this person be depressed at Wave 8?" |
| **Outcome** | CES-D ≥ 3 at W8 | Standard 8-item ELSA threshold (Steffick, 2000) |
| **Predictors** | W6 + W7 features | 2–4 year lead time = "early prediction" |
| **Models** | Logistic Regression + Random Forest | Interpretable baseline vs non-linear comparison |
| **Evaluation** | ROC-AUC (primary), F1, Precision, Recall, Confusion Matrix, PR curve |
| **Validation** | Stratified 5-fold CV + 25% held-out test set |

### Notebook sections

1. **Data Loading & Merging** — Load IFS derived + core wave + financial files; merge on `idauniq`
2. **Preprocessing** — Handle survey missing codes, imputation, encoding
3. **Feature Engineering** — Delta CES-D, mobility/ADL counts, domain grouping
4. **Modelling** — Logistic Regression + Random Forest with stratified CV
5. **Evaluation** — Full metrics, ROC/PR curves, confusion matrices
6. **Interpretation** — Feature importance + SHAP
7. **Leakage Sensitivity** — With vs without prior CES-D

---
## 0. Setup

In [ ]:
import sys, os, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score, accuracy_score
)
import shap

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate repo root from {Path.cwd()} — ensure config.py exists at root"
    )
sys.path.insert(0, str(REPO_ROOT))
from config import DATA_ROOT, CORE, IFS, FIN, OUTPUTS, check_paths

FIG_DIR = OUTPUTS / "figures"
RES_DIR = OUTPUTS / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
CESD_CUTOFF = 3

check_paths()
print(f"\nCES-D cut-off: >= {CESD_CUTOFF}")
print(f"Figures  -> {FIG_DIR}")
print(f"Results  -> {RES_DIR}")

---
## 1. Data Loading & Merging

We load three file types per wave:
- **IFS derived** — pre-validated summary variables (demographics, CES-D score, cognition, etc.)
- **Core interview** — raw questionnaire items (physical activity, chronic conditions, social contact)
- **Financial derived** — income/wealth quintiles

All files are merged on `idauniq` (unique participant ID). We keep only participants present in **all three waves** (inner join) to form a balanced longitudinal panel.

In [ ]:
# --- IFS derived variables (pre-validated by IFS research team) ---
IFS_COLS = [
    "idauniq", "age", "sex", "nonwhite", "marstat", "couple",
    "srh_hrs", "llsill", "hlimwrk",
    "hemobwa", "hemobsi", "hemobch", "hemobcs", "hemobcl",
    "hemobst", "hemobre", "hemobpu", "hemobli", "hemobpi", "hemob96",
    "headldr", "headlwa", "headlba", "headlea", "headlbe", "headlwc",
    "headlma", "headlda", "headlpr", "headlsh", "headlph",
    "headlco", "headlme", "headlho", "headlmo", "headl96",
    "memtotb", "execnn",
    "ecpos", "qual3",
    "findiff", "ndepriv", "lackresb",
    "famtype", "tenure", "nsibs", "ngrandch",
    "smokerstat",
    "cesd_sc", "cesd_na",
]

# --- Core interview variables (raw questionnaire) ---
CORE_COLS = [
    "idauniq",
    "Hehelf", "Heill",
    "HeActa", "HeActb", "HeActc",
    "hediabp", "hediadi", "hediami", "hediast", "hediahf",
    "hediaar", "hediaan",
    "scint",
    "chinhh", "chouthh",
]

# --- Financial derived (wealth/income quintiles) ---
FIN_COLS = ["idauniq", "totwq5_bu_s"]

def load_wave(wave: int, suffix: str) -> pd.DataFrame:
    """Load and merge IFS + core + financial for one wave, adding _wN suffix."""
    ifs = pd.read_stata(str(IFS[wave]), convert_categoricals=False, columns=IFS_COLS)
    core = pd.read_stata(str(CORE[wave]), convert_categoricals=False, columns=CORE_COLS)
    fin = pd.read_stata(str(FIN[wave]), convert_categoricals=False, columns=FIN_COLS)

    df = ifs.merge(core, on="idauniq", how="inner", suffixes=("", "_core"))
    df = df.merge(fin, on="idauniq", how="inner")

    # Drop any duplicate columns from merge
    df = df.loc[:, ~df.columns.duplicated()]

    rename = {c: f"{c}_{suffix}" for c in df.columns if c != "idauniq"}
    df = df.rename(columns=rename)
    print(f"  Wave {wave}: {len(df)} rows, {len(df.columns)-1} features")
    return df


print("Loading waves...")
w6 = load_wave(6, "w6")
w7 = load_wave(7, "w7")

# Wave 8: only need the outcome (CES-D) from IFS
w8 = pd.read_stata(str(IFS[8]), convert_categoricals=False, columns=["idauniq", "cesd_sc"])
w8 = w8.rename(columns={"cesd_sc": "cesd_sc_w8"})
print(f"  Wave 8 outcome: {len(w8)} rows")

# Inner merge — only people in all three waves
panel = w6.merge(w7, on="idauniq", how="inner").merge(w8, on="idauniq", how="inner")
print(f"\nPanel (all three waves): {len(panel)} participants")

---
## 1b. Outcome Variable Construction

We use the **validated IFS-derived `cesd_sc`** (0–8 scale, sum of 8 binary CES-D items) rather than reconstructing from raw PSced items. This avoids the scoring artefact found in the earlier preprocessing notebook.

**Threshold:** CES-D ≥ 3 = depressed (standard ELSA cut-off; Steffick, 2000).

In [ ]:
SURVEY_MISSING = {-1, -2, -8, -9}

# Drop rows where W8 CES-D is a survey missing code or NaN
panel = panel[~panel["cesd_sc_w8"].isin(SURVEY_MISSING)].copy()
panel = panel.dropna(subset=["cesd_sc_w8"])

# Binary outcome
panel["depressed_w8"] = (panel["cesd_sc_w8"] >= CESD_CUTOFF).astype(int)

n_dep = panel["depressed_w8"].sum()
n_not = len(panel) - n_dep
prev = n_dep / len(panel)

print(f"Analytic sample: {len(panel)} participants")
print(f"  Depressed (CES-D >= {CESD_CUTOFF}): {n_dep} ({prev:.1%})")
print(f"  Not depressed:                     {n_not} ({1-prev:.1%})")
print(f"  Class ratio:                       1:{n_not/n_dep:.1f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
panel["cesd_sc_w8"].hist(bins=9, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].axvline(CESD_CUTOFF, color="red", ls="--", label=f"Cut-off = {CESD_CUTOFF}")
axes[0].set_xlabel("CES-D Score (W8)")
axes[0].set_ylabel("Count")
axes[0].set_title("CES-D Distribution at Wave 8")
axes[0].legend()

panel["depressed_w8"].value_counts().sort_index().plot.bar(
    ax=axes[1], color=["steelblue", "coral"], edgecolor="white"
)
axes[1].set_xticklabels(["Not Depressed (0)", "Depressed (1)"], rotation=0)
axes[1].set_ylabel("Count")
axes[1].set_title("Class Distribution")

plt.tight_layout()
plt.savefig(FIG_DIR / "outcome_distribution.png", bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'outcome_distribution.png'}")

---
## 2. Preprocessing

### Steps:
1. **Recode survey missing codes** (-1, -2, -8, -9) → NaN across all predictors
2. **Drop near-empty columns** (>40% missing after recoding)
3. **Coerce all predictors to numeric** (ELSA variables are integer-coded; sklearn handles the rest)
4. Imputation and scaling happen inside sklearn Pipelines (Section 4) to prevent data leakage

In [ ]:
# Identify predictor columns (W6 and W7 only — never W8 except the label)
exclude = {"idauniq", "cesd_sc_w8", "depressed_w8"}
predictor_cols = [c for c in panel.columns if c not in exclude]

print(f"Total predictor candidates: {len(predictor_cols)}")

# 1. Recode survey missing codes to NaN
for col in predictor_cols:
    panel[col] = pd.to_numeric(panel[col], errors="coerce")
    panel.loc[panel[col].isin(SURVEY_MISSING), col] = np.nan

# 2. Drop columns with >40% missing (computed on training split only to avoid leakage)
from sklearn.model_selection import train_test_split as _tts
_Xm, _, _, _ = _tts(panel[predictor_cols], panel["depressed_w8"],
                     test_size=0.25, stratify=panel["depressed_w8"], random_state=SEED)
miss_pct = _Xm.isna().mean().sort_values(ascending=False)
high_miss = miss_pct[miss_pct > 0.40].index.tolist()
print(f"Dropping {len(high_miss)} columns with >40% missing (train-only estimate): {high_miss}")

predictor_cols = [c for c in predictor_cols if c not in high_miss]
print(f"Remaining predictors: {len(predictor_cols)}")

# 3. Show missingness summary for remaining columns
miss_remaining = panel[predictor_cols].isna().mean().sort_values(ascending=False)
print(f"\nMax missingness among remaining: {miss_remaining.iloc[0]:.1%} ({miss_remaining.index[0]})")
print(f"Median missingness: {miss_remaining.median():.2%}")

# Visualise top-20 missingness
top_miss = miss_remaining.head(20)
if top_miss.max() > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    top_miss.plot.barh(ax=ax, color="coral")
    ax.set_xlabel("Fraction Missing")
    ax.set_title("Top 20 Predictors by Missingness")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "missingness_top20.png", bbox_inches="tight")
    plt.show()

---
## 3. Feature Engineering

We construct clinically meaningful derived features from the raw predictors:

| Feature | Definition | Rationale |
|---------|------------|-----------|
| `cesd_change` | CES-D(W7) − CES-D(W6) | Worsening trajectory predicts future depression |
| `mobility_count_wN` | Sum of 10 mobility difficulty items | Composite physical function score |
| `adl_count_wN` | Sum of 6 ADL difficulty items | Activities of daily living burden |
| `iadl_count_wN` | Sum of 10 IADL difficulty items | Instrumental ADL burden |
| `functional_burden_wN` | ADL + IADL count | Overall functional impairment |

In [ ]:
MOBILITY_ITEMS = ["hemobwa", "hemobsi", "hemobch", "hemobcs", "hemobcl",
                  "hemobst", "hemobre", "hemobpu", "hemobli", "hemobpi"]
ADL_ITEMS = ["headldr", "headlwa", "headlba", "headlea", "headlbe", "headlwc"]
IADL_ITEMS = ["headlma", "headlda", "headlpr", "headlsh", "headlph",
              "headlco", "headlme", "headlho", "headlmo"]

def safe_sum(df, items, suffix):
    cols = [f"{item}_{suffix}" for item in items]
    present = [c for c in cols if c in df.columns]
    if not present:
        return pd.Series(np.nan, index=df.index)
    return df[present].sum(axis=1, min_count=1)

for sfx in ["w6", "w7"]:
    panel[f"mobility_count_{sfx}"] = safe_sum(panel, MOBILITY_ITEMS, sfx)
    panel[f"adl_count_{sfx}"] = safe_sum(panel, ADL_ITEMS, sfx)
    panel[f"iadl_count_{sfx}"] = safe_sum(panel, IADL_ITEMS, sfx)
    panel[f"functional_burden_{sfx}"] = panel[f"adl_count_{sfx}"] + panel[f"iadl_count_{sfx}"]

# CES-D trajectory (temporal change)
panel["cesd_change"] = panel["cesd_sc_w7"] - panel["cesd_sc_w6"]

# Mobility change
panel["mobility_change"] = panel["mobility_count_w7"] - panel["mobility_count_w6"]

# Update predictor list
engineered = [
    "mobility_count_w6", "mobility_count_w7",
    "adl_count_w6", "adl_count_w7",
    "iadl_count_w6", "iadl_count_w7",
    "functional_burden_w6", "functional_burden_w7",
    "cesd_change", "mobility_change",
]
predictor_cols = predictor_cols + engineered
predictor_cols = list(dict.fromkeys(predictor_cols))  # deduplicate preserving order

print(f"Total predictors after engineering: {len(predictor_cols)}")
print(f"\nEngineered features:")
for e in engineered:
    vals = panel[e].dropna()
    print(f"  {e}: mean={vals.mean():.2f}, std={vals.std():.2f}, "
          f"range=[{vals.min():.0f}, {vals.max():.0f}], missing={panel[e].isna().mean():.1%}")

### 3b. Feature Domain Summary

Group predictors by clinical domain for interpretability in the presentation.

In [ ]:
DOMAINS = {
    "Prior Depression": [c for c in predictor_cols if "cesd" in c.lower()],
    "Demographics": [c for c in predictor_cols if any(c.startswith(p) for p in
                     ["age_", "sex_", "nonwhite_", "marstat_", "couple_"])],
    "Self-rated Health": [c for c in predictor_cols if any(p in c.lower() for p in
                          ["hehelf", "heill", "srh_hrs", "llsill", "hlimwrk"])],
    "Chronic Conditions": [c for c in predictor_cols if c.lower().startswith("hedia")],
    "Mobility & Function": [c for c in predictor_cols if any(p in c.lower() for p in
                            ["mobility", "adl_count", "iadl_count", "functional_burden",
                             "hemob96", "headl96"])],
    "Cognition": [c for c in predictor_cols if any(p in c for p in ["memtotb", "execnn"])],
    "Socioeconomic": [c for c in predictor_cols if any(p in c.lower() for p in
                      ["ecpos", "qual3", "findiff", "lackres", "totwq5", "wpact"])],
    "Social & Housing": [c for c in predictor_cols if any(p in c.lower() for p in
                         ["famtype", "tenure", "nsibs", "ngrandch", "chinhh",
                          "chouthh", "smokerstat"])],
    "Lifestyle & Digital": [c for c in predictor_cols if any(p in c.lower() for p in
                            ["heact", "scint", "scptr"])],
}

assigned = set()
for domain, cols in DOMAINS.items():
    assigned.update(cols)
    print(f"{domain}: {len(cols)} features")

unassigned = [c for c in predictor_cols if c not in assigned]
if unassigned:
    DOMAINS["Other"] = unassigned
    print(f"Other (unassigned): {len(unassigned)} features")

total_assigned = sum(len(v) for v in DOMAINS.values())
print(f"\nTotal features across domains: {total_assigned}")

---
## 4. Model Training

### Strategy:
1. **75/25 stratified split** → train and held-out test sets
2. **Stratified 5-fold CV** on the training set for robust performance estimates
3. **Final evaluation** on the held-out test set (never seen during training or CV)

Both models use `class_weight='balanced'` to handle the ~80/20 class imbalance. Imputation (median) and scaling (standard) are embedded in sklearn Pipelines to prevent data leakage.

In [ ]:
X = panel[predictor_cols].copy()
y = panel["depressed_w8"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

print(f"Training set: {len(X_train)} ({y_train.mean():.1%} depressed)")
print(f"Test set:     {len(X_test)} ({y_test.mean():.1%} depressed)")

# Define models
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=5000, class_weight="balanced",
            random_state=SEED, solver="lbfgs", C=1.0
        )),
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=300, class_weight="balanced",
            random_state=SEED, n_jobs=-1, max_depth=15,
            min_samples_leaf=10
        )),
    ]),
}

### 4b. Stratified 5-Fold Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_scoring = ["roc_auc", "f1", "precision", "recall", "accuracy"]

cv_results = {}
for name, pipe in models.items():
    print(f"\n{'='*50}")
    print(f"Cross-validating: {name}")
    print(f"{'='*50}")

    scores = cross_validate(
        pipe, X_train, y_train, cv=cv, scoring=cv_scoring,
        return_train_score=False, n_jobs=-1
    )
    cv_results[name] = scores

    for metric in cv_scoring:
        key = f"test_{metric}"
        vals = scores[key]
        print(f"  {metric:>12}: {vals.mean():.4f} (+/- {vals.std():.4f})")

# Summary table
cv_summary = []
for name, scores in cv_results.items():
    row = {"Model": name}
    for metric in cv_scoring:
        vals = scores[f"test_{metric}"]
        row[f"{metric}_mean"] = vals.mean()
        row[f"{metric}_std"] = vals.std()
    cv_summary.append(row)

cv_df = pd.DataFrame(cv_summary)
print("\n" + "="*60)
print("Cross-Validation Summary")
print("="*60)
display(cv_df.round(4))

### 4c. Final Model Fitting on Full Training Set

In [ ]:
fitted_models = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    print(f"{name}: fitted on {len(X_train)} samples")

print("\nAll models trained. Ready for evaluation on held-out test set.")

---
## 5. Evaluation on Held-Out Test Set

This section reports all metrics on the **test set** (data never seen during training or CV).

In [ ]:
test_results = []

for name, pipe in fitted_models.items():
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    row = {
        "Model": name,
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "F1": f1_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "Accuracy": accuracy_score(y_test, y_pred),
        "Avg Precision": average_precision_score(y_test, y_proba),
    }
    test_results.append(row)

    print(f"\n{'='*50}")
    print(f"{name} — Test Set Performance")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=["Not Depressed", "Depressed"]))

results_df = pd.DataFrame(test_results).set_index("Model")
results_df.to_csv(RES_DIR / "test_set_results.csv")
print("\n" + "="*60)
print("Test Set Results Summary")
print("="*60)
display(results_df.round(4))
print(f"\nSaved: {RES_DIR / 'test_set_results.csv'}")

### 5b. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for name, pipe in fitted_models.items():
    y_proba = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_val = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.3f})", linewidth=2)

ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random (AUC = 0.500)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Test Set")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIG_DIR / "roc_curves.png", bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'roc_curves.png'}")

### 5c. Precision-Recall Curves

With ~19% prevalence, PR curves are more informative than ROC for the minority class.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for name, pipe in fitted_models.items():
    y_proba = pipe.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    ax.plot(rec, prec, label=f"{name} (AP = {ap:.3f})", linewidth=2)

baseline = y_test.mean()
ax.axhline(baseline, color="grey", ls="--", alpha=0.5, label=f"Baseline ({baseline:.2f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — Test Set")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(FIG_DIR / "pr_curves.png", bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'pr_curves.png'}")

### 5d. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, pipe) in zip(axes, fitted_models.items()):
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Not Dep.", "Depressed"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)

plt.suptitle("Confusion Matrices — Test Set", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrices.png", bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'confusion_matrices.png'}")

---
## 6. Interpretation — Feature Importance & SHAP

Understanding **which predictors drive the model** is essential for the presentation and for clinical relevance. We use:
- **Logistic Regression coefficients** (standardised log-odds; per 1 SD change after scaling)
- **Random Forest feature importance** (Gini impurity-based)
- **SHAP values** on the Random Forest (model-agnostic explanations)

In [ ]:
# --- Logistic Regression Coefficients (top 20 by absolute value) ---
lr_pipe = fitted_models["Logistic Regression"]
lr_clf = lr_pipe.named_steps["clf"]
coef_df = pd.DataFrame({
    "Feature": predictor_cols,
    "Coefficient": lr_clf.coef_[0]
}).sort_values("Coefficient", key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(9, 8))
top20 = coef_df.head(20)
colors = ["coral" if v > 0 else "steelblue" for v in top20["Coefficient"]]
ax.barh(top20["Feature"], top20["Coefficient"], color=colors)
ax.set_xlabel("Standardised Coefficient (log-odds per 1 SD)")
ax.set_title("Logistic Regression — Top 20 Predictors (standardised)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIG_DIR / "lr_coefficients_top20.png", bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'lr_coefficients_top20.png'}")

In [ ]:
# --- Random Forest Feature Importance (top 20) ---
rf_pipe = fitted_models["Random Forest"]
rf_clf = rf_pipe.named_steps["clf"]
importance_df = pd.DataFrame({
    "Feature": predictor_cols,
    "Importance": rf_clf.feature_importances_
}).sort_values("Importance", ascending=False)

fig, ax = plt.subplots(figsize=(9, 8))
top20_rf = importance_df.head(20)
ax.barh(top20_rf["Feature"], top20_rf["Importance"], color="steelblue")
ax.set_xlabel("Feature Importance (Gini)")
ax.set_title("Random Forest — Top 20 Predictors")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIG_DIR / "rf_importance_top20.png", bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'rf_importance_top20.png'}")

In [ ]:
# --- SHAP Analysis (Random Forest) ---
# Impute test data the same way the pipeline does, for SHAP
imputer = rf_pipe.named_steps["imputer"]
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test), columns=predictor_cols, index=X_test.index
)

# Use a sample for speed if dataset is large
shap_sample = X_test_imputed.sample(n=min(500, len(X_test_imputed)), random_state=SEED)

explainer = shap.TreeExplainer(rf_clf)
explanation = explainer(shap_sample)

# For binary classification: explanation.values has shape (n, features, 2)
# Take the positive class (depressed = class 1)
if explanation.values.ndim == 3:
    shap_vals = explanation.values[:, :, 1]
else:
    shap_vals = explanation.values

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_vals, shap_sample, max_display=20, show=False, plot_size=None)
plt.title("SHAP Summary — Random Forest (Top 20)", fontsize=13, pad=15)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_summary_rf.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Saved: {FIG_DIR / 'shap_summary_rf.png'}")

In [ ]:
# SHAP bar plot (mean |SHAP|)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_vals, shap_sample, plot_type="bar", max_display=20, show=False, plot_size=None)
plt.title("Mean |SHAP| — Random Forest (Top 20)", fontsize=13, pad=15)
plt.xlabel("Mean |SHAP value|")
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_bar_rf.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Saved: {FIG_DIR / 'shap_bar_rf.png'}")

---
## 7. Leakage Sensitivity Analysis

**Question:** How much of our model's predictive power comes from **prior depression scores** (CES-D at W6 and W7) vs other domains (health, social, economic)?

If removing prior CES-D drastically drops performance, it means the model is mostly saying "people who were depressed stay depressed" — useful but not novel. If performance stays reasonable, the non-depression features have independent predictive value.

This is critical for the **limitations** section of the presentation.

In [ ]:
PRIOR_CESD_COLS = [c for c in predictor_cols if "cesd" in c.lower()]
predictors_no_cesd = [c for c in predictor_cols if c not in PRIOR_CESD_COLS]

print(f"Prior CES-D columns removed: {PRIOR_CESD_COLS}")
print(f"Predictors without CES-D: {len(predictors_no_cesd)} (was {len(predictor_cols)})")

X_train_nc = X_train[predictors_no_cesd]
X_test_nc = X_test[predictors_no_cesd]

sensitivity_results = []

for name_base, pipe_factory in [
    ("Logistic Regression", lambda: Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=5000, class_weight="balanced",
            random_state=SEED, solver="lbfgs"
        )),
    ])),
    ("Random Forest", lambda: Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=300, class_weight="balanced",
            random_state=SEED, n_jobs=-1, max_depth=15,
            min_samples_leaf=10
        )),
    ])),
]:
    for label, X_tr, X_te, incl_cesd in [
        ("Full (with CES-D)", X_train, X_test, "Yes"),
        ("No prior CES-D", X_train_nc, X_test_nc, "No"),
    ]:
        pipe = pipe_factory()
        pipe.fit(X_tr, y_train)
        y_pred = pipe.predict(X_te)
        y_proba = pipe.predict_proba(X_te)[:, 1]

        sensitivity_results.append({
            "Model": name_base,
            "Feature Set": label,
            "Prior CES-D": incl_cesd,
            "N Features": X_tr.shape[1],
            "ROC-AUC": roc_auc_score(y_test, y_proba),
            "F1": f1_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred),
        })

sens_df = pd.DataFrame(sensitivity_results)
sens_df.to_csv(RES_DIR / "leakage_sensitivity.csv", index=False)
print("\n" + "="*70)
print("Leakage Sensitivity: With vs Without Prior CES-D")
print("="*70)
display(sens_df.round(4))
print(f"\nSaved: {RES_DIR / 'leakage_sensitivity.csv'}")

In [ ]:
# Visualise the leakage sensitivity
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, metric in zip(axes, ["ROC-AUC", "F1"]):
    pivot = sens_df.pivot(index="Model", columns="Feature Set", values=metric)
    pivot.plot.bar(ax=ax, color=["steelblue", "coral"], edgecolor="white", width=0.6)
    ax.set_ylabel(metric)
    ax.set_title(f"{metric}: With vs Without Prior CES-D")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.legend(title="Feature Set")
    ax.set_ylim(0, 1)

plt.suptitle("Leakage Sensitivity Analysis", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "leakage_sensitivity.png", bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'leakage_sensitivity.png'}")

---
## 8. Summary & Export

### Key outputs saved to `outputs/`:

| File | Content |
|------|---------|
| `results/test_set_results.csv` | Primary model comparison metrics |
| `results/leakage_sensitivity.csv` | With vs without prior CES-D |
| `figures/outcome_distribution.png` | CES-D histogram + class bar chart |
| `figures/roc_curves.png` | ROC curves for both models |
| `figures/pr_curves.png` | Precision-Recall curves |
| `figures/confusion_matrices.png` | Side-by-side confusion matrices |
| `figures/lr_coefficients_top20.png` | Top LR coefficients |
| `figures/rf_importance_top20.png` | Top RF feature importances |
| `figures/shap_summary_rf.png` | SHAP beeswarm plot |
| `figures/shap_bar_rf.png` | SHAP mean absolute bar chart |
| `figures/leakage_sensitivity.png` | Leakage sensitivity comparison |

In [ ]:
# Final summary print
print("=" * 65)
print("ELSA Depression Prediction — Pipeline Complete")
print("=" * 65)
print(f"")
print(f"  Analytic sample:  {len(panel)} participants (W6 + W7 + W8)")
print(f"  Outcome:          CES-D >= {CESD_CUTOFF} at Wave 8 (prevalence: {prev:.1%})")
print(f"  Predictors:       {len(predictor_cols)} features from W6 and W7")
print(f"  Train / Test:     {len(X_train)} / {len(X_test)}")
print()
print("  Test Set Results:")
for _, row in results_df.iterrows():
    print(f"    {row.name}: AUC={row['ROC-AUC']:.3f}, F1={row['F1']:.3f}, "
          f"Precision={row['Precision']:.3f}, Recall={row['Recall']:.3f}")
print()
print(f"  Figures saved to: {FIG_DIR}")
print(f"  Results saved to: {RES_DIR}")
print("=" * 65)

# Save full predictor list
pred_export = pd.DataFrame({
    "feature": predictor_cols,
    "in_model": True
})
pred_export.to_csv(RES_DIR / "predictor_list.csv", index=False)
print(f"Saved: {RES_DIR / 'predictor_list.csv'}")